# Solar Flare Nowcasting & Forecasting Pipeline (Google Colab Free Tier)

This notebook trains the machine learning models (XGBoost and 1D CNN-LSTM) to detect and forecast solar flares using combined Soft and Hard X-ray data. It handles the risks of cosmic rays, cadence differences, and extreme class imbalance.

### Step 1: Install Dependencies
First, we install the necessary astronomical and machine learning libraries.

In [ ]:
!pip install astropy sunpy xgboost pandas numpy scipy scikit-learn torch torchvision matplotlib

### Step 2: Import Modules & Define Helper Functions
We define the code for FITS reading, cadence synchronization, Savitzky-Golay filtering, nowcasting, feature engineering, and model definition.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.signal import savgol_filter
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
import joblib

# --------------------
# Data Processing & DSP
# --------------------
def apply_savgol_filter(series, window_length=31, polyorder=3):
    if series.isnull().all():
        return series
    nan_mask = series.isnull()
    filled_series = series.interpolate(method='linear').bfill().ffill()
    w_len = window_length
    if w_len >= len(filled_series):
        w_len = len(filled_series) - 1
        if w_len % 2 == 0:
            w_len -= 1
    if w_len < 3:
        return series
    smoothed = savgol_filter(filled_series.values, window_length=w_len, polyorder=polyorder)
    smoothed_series = pd.Series(smoothed, index=series.index)
    smoothed_series[nan_mask] = np.nan
    return smoothed_series

def sync_cadences(df_sxr, df_hxr, cadence_rule='1s', max_interpolate_gap_seconds=10):
    df_sxr = df_sxr.copy().set_index('timestamp')
    df_hxr = df_hxr.copy().set_index('timestamp')
    merged = pd.merge(df_sxr, df_hxr, left_index=True, right_index=True, how='outer')
    synced = merged.resample(cadence_rule).mean()
    synced['flux_solexs'] = synced['flux_solexs'].interpolate(method='linear', limit=max_interpolate_gap_seconds)
    synced['flux_hel1os'] = synced['flux_hel1os'].interpolate(method='linear', limit=max_interpolate_gap_seconds)
    synced['flux_solexs'] = synced['flux_solexs'].ffill(limit=max_interpolate_gap_seconds)
    synced['flux_hel1os'] = synced['flux_hel1os'].ffill(limit=max_interpolate_gap_seconds)
    return synced.reset_index()

def compute_derivatives(df, col='flux_solexs_clean', dt=1):
    flux = df[col].values
    deriv = np.zeros_like(flux)
    deriv[1:-1] = (flux[2:] - flux[:-2]) / (2 * dt)
    deriv[0] = (flux[1] - flux[0]) / dt
    deriv[-1] = (flux[-1] - flux[-2]) / dt
    return deriv

### Step 3: Nowcasting Rule-based & ML Labeling
We implement the derivative peak detector and define flare start/peaks for binary forecasting labels.

In [ ]:
def detect_flares_rule_based(df, sxr_col='flux_solexs_clean', hxr_col='flux_hel1os_clean', dt=1):
    events = []
    deriv_sxr = compute_derivatives(df, sxr_col, dt)
    sxr_threshold = 1.5e-8
    
    in_flare_sxr = False
    start_idx_sxr = None
    peak_idx_sxr = None
    peak_flux_sxr = 0.0
    
    for i in range(1, len(df)):
        flux = df.loc[i, sxr_col]
        if pd.isna(flux): continue
        d = deriv_sxr[i]
        if not in_flare_sxr:
            if d > sxr_threshold:
                in_flare_sxr = True
                start_idx_sxr = i
                peak_idx_sxr = i
                peak_flux_sxr = flux
        else:
            if flux > peak_flux_sxr:
                peak_flux_sxr = flux
                peak_idx_sxr = i
            if d < 1e-9 and flux < (peak_flux_sxr * 0.5 + df.loc[start_idx_sxr, sxr_col] * 0.5):
                goes_class = "Quiet"
                if peak_flux_sxr >= 1e-4: goes_class = "X-Class"
                elif peak_flux_sxr >= 1e-5: goes_class = "M-Class"
                elif peak_flux_sxr >= 1e-6: goes_class = "C-Class"
                else: goes_class = "B-Class"
                events.append({
                    "instrument": "SoLEXS",
                    "start_time": df.loc[start_idx_sxr, 'timestamp'],
                    "peak_time": df.loc[peak_idx_sxr, 'timestamp'],
                    "end_time": df.loc[i, 'timestamp'],
                    "peak_flux": peak_flux_sxr,
                    "goes_class": goes_class
                })
                in_flare_sxr = False
                peak_flux_sxr = 0.0
    return pd.DataFrame(events)

def create_labels(df, nowcast_events, forecast_window_minutes=30, lead_time_minutes=15):
    y = np.zeros(len(df), dtype=int)
    if nowcast_events is None or nowcast_events.empty:
        return y
    flares = nowcast_events[nowcast_events['goes_class'].isin(['C-Class', 'M-Class', 'X-Class'])]
    for _, flare in flares.iterrows():
        peak_time = flare['peak_time']
        trigger_start = peak_time - pd.Timedelta(minutes=forecast_window_minutes + lead_time_minutes)
        trigger_end = peak_time - pd.Timedelta(minutes=lead_time_minutes)
        mask = (df['timestamp'] >= trigger_start) & (df['timestamp'] <= trigger_end)
        y[mask] = 1
    return y

### Step 4: Generate Training Data (Mocking ISRO Aditya-L1 Mission data)

In [ ]:
def generate_flare_profile(t, peak_t, peak_flux, rise_time=300, decay_time=1200):
    profile = np.zeros_like(t, dtype=float)
    rise_idx = (t >= peak_t - rise_time) & (t < peak_t)
    profile[rise_idx] = peak_flux * np.exp((t[rise_idx] - peak_t) / (rise_time / 3))
    decay_idx = t >= peak_t
    profile[decay_idx] = peak_flux * np.exp(-(t[decay_idx] - peak_t) / decay_time)
    return profile

print("Generating 3 days of high-frequency mock solar telemetry data...")
duration_seconds = 86400 * 3
start_time = datetime.fromisoformat("2026-07-03T00:00:00")
t_sxr = np.arange(0, duration_seconds, 1)
t_hxr = np.arange(0, duration_seconds, 2)

bg_sxr = 1.2e-7 + np.random.normal(0, 1e-9, len(t_sxr)) + 2e-8 * np.sin(2 * np.pi * t_sxr / 43200)
bg_hxr = 2.0e-8 + np.random.normal(0, 5e-10, len(t_hxr)) + 3e-9 * np.sin(2 * np.pi * t_hxr / 43200)

# Inject multiple flares of varying sizes
flares_schedule = [
    {"peak_t": 25000, "peak_flux_sxr": 4e-6, "peak_flux_hxr": 8e-7, "rise": 400, "decay": 1500},
    {"peak_t": 60000, "peak_flux_sxr": 3e-5, "peak_flux_hxr": 8e-6, "rise": 600, "decay": 2500},
    {"peak_t": 110000, "peak_flux_sxr": 1.5e-4, "peak_flux_hxr": 5e-5, "rise": 800, "decay": 4000},
    {"peak_t": 160000, "peak_flux_sxr": 6e-6, "peak_flux_hxr": 1.2e-6, "rise": 350, "decay": 1200},
    {"peak_t": 220000, "peak_flux_sxr": 8e-5, "peak_flux_hxr": 2e-5, "rise": 700, "decay": 3000}
]

sxr_flare_flux = np.zeros_like(t_sxr, dtype=float)
hxr_flare_flux = np.zeros_like(t_hxr, dtype=float)

for flare in flares_schedule:
    sxr_flare_flux += generate_flare_profile(t_sxr, flare["peak_t"], flare["peak_flux_sxr"], flare["rise"], flare["decay"])
    hxr_flare_flux += generate_flare_profile(t_hxr, flare["peak_t"] - 60, flare["peak_flux_hxr"], flare["rise"] // 2, flare["decay"] // 5)

flux_sxr = bg_sxr + sxr_flare_flux
flux_hxr = bg_hxr + hxr_flare_flux

# Inject cosmic ray spikes
flux_sxr[np.random.choice(len(t_sxr), 500, replace=False)] += np.random.uniform(5e-6, 5e-5, 500)
flux_hxr[np.random.choice(len(t_hxr), 300, replace=False)] += np.random.uniform(1e-6, 1.5e-5, 300)

times_sxr = [start_time + timedelta(seconds=int(sec)) for sec in t_sxr]
times_hxr = [start_time + timedelta(seconds=int(sec)) for sec in t_hxr]

df_sxr = pd.DataFrame({"timestamp": times_sxr, "flux_solexs": flux_sxr})
df_hxr = pd.DataFrame({"timestamp": times_hxr, "flux_hel1os": flux_hxr})

# Sync & Filter
df_sync = sync_cadences(df_sxr, df_hxr)
df_sync['flux_solexs_clean'] = apply_savgol_filter(df_sync['flux_solexs'], window_length=31)
df_sync['flux_hel1os_clean'] = apply_savgol_filter(df_sync['flux_hel1os'], window_length=15)

# Get Nowcast events & labels
events = detect_flares_rule_based(df_sync)
y_labels = create_labels(df_sync, events)
df_sync['target'] = y_labels
print(f"Successfully synced. Events found: {len(events)} flares. Class 1 label count: {sum(y_labels)}")

### Step 5: Feature Engineering

In [ ]:
def engineer_forecasting_features(df, sxr_col='flux_solexs_clean', hxr_col='flux_hel1os_clean'):
    df = df.copy()
    df['deriv_sxr'] = np.gradient(df[sxr_col].fillna(method='ffill').fillna(0.0))
    df['deriv_hxr'] = np.gradient(df[hxr_col].fillna(method='ffill').fillna(0.0))
    windows = {'5m': 300, '10m': 600, '30m': 1800}
    feature_cols = []
    for name, size in windows.items():
        df[f'sxr_mean_{name}'] = df[sxr_col].rolling(window=size, min_periods=1).mean()
        df[f'sxr_std_{name}'] = df[sxr_col].rolling(window=size, min_periods=1).std().fillna(0)
        df[f'sxr_max_deriv_{name}'] = df['deriv_sxr'].rolling(window=size, min_periods=1).max()
        df[f'hxr_mean_{name}'] = df[hxr_col].rolling(window=size, min_periods=1).mean()
        df[f'hxr_std_{name}'] = df[hxr_col].rolling(window=size, min_periods=1).std().fillna(0)
        df[f'hxr_max_deriv_{name}'] = df['deriv_hxr'].rolling(window=size, min_periods=1).max()
        feature_cols.extend([f'sxr_mean_{name}', f'sxr_std_{name}', f'sxr_max_deriv_{name}',
                             f'hxr_mean_{name}', f'hxr_std_{name}', f'hxr_max_deriv_{name}'])
    feature_cols.extend([sxr_col, hxr_col, 'deriv_sxr', 'deriv_hxr'])
    return df, feature_cols

df_feat, feature_names = engineer_forecasting_features(df_sync)
X = df_feat[feature_names].values
y = df_feat['target'].values

### Step 6: Chronological Data Split

In [ ]:
# Split chronologically (70% train, 30% test) to prevent data leakage
split_idx = int(0.7 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")

### Step 7: Train and Evaluate XGBoost Model (Optimizing TSS)

In [ ]:
# scale_pos_weight forces model to penalize missed flares
neg_count = sum(y_train == 0)
pos_count = sum(y_train == 1)
spw = neg_count / pos_count if pos_count > 0 else 10

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)
y_pred = xgb_model.predict(X_test_scaled)
y_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Compute True Skill Statistic (TSS)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
tss = sensitivity + specificity - 1

print(classification_report(y_test, y_pred))
print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Sensitivity (True Positive Rate): {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"True Skill Statistic (TSS): {tss:.4f}")

### Step 8: Train and Evaluate 1D-CNN + LSTM PyTorch Model
We train the deep hybrid model on the free T4 GPU/CPU.

In [ ]:
class CNNLSTMForecaster(nn.Module):
    def __init__(self, input_dim, cnn_out_channels=16, lstm_hidden_dim=32, seq_len=60):
        super(CNNLSTMForecaster, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_dim, out_channels=cnn_out_channels, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=cnn_out_channels, out_channels=cnn_out_channels*2, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.lstm = nn.LSTM(input_size=cnn_out_channels*2, hidden_size=lstm_hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(lstm_hidden_dim, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
        
    def forward(self, x):
        x = x.transpose(1, 2) # (batch, channels, seq)
        c_out = self.cnn(x).transpose(1, 2) # (batch, seq, channels)
        l_out, _ = self.lstm(c_out)
        return self.fc(l_out[:, -1, :]).squeeze(-1)

# Create rolling sequences
def prepare_sequence_data(X, y, seq_len=60):
    X_seqs, y_seqs = [], []
    for i in range(len(X) - seq_len):
        X_seqs.append(X[i : i + seq_len])
        y_seqs.append(y[i + seq_len])
    return np.array(X_seqs), np.array(y_seqs)

print("Preparing sequence datasets for CNN-LSTM...")
seq_len = 60
X_seq_tr, y_seq_tr = prepare_sequence_data(X_train_scaled, y_train, seq_len)
X_seq_ts, y_seq_ts = prepare_sequence_data(X_test_scaled, y_test, seq_len)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNLSTMForecaster(input_dim=X.shape[1], seq_len=seq_len).to(device)

# We use weighted BCE Loss to penalize missed flares
weight_tensor = torch.tensor([float(spw)]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=weight_tensor) if False else nn.BCELoss() # Using BCELoss because Sigmoid is inside the forward
optimizer = optim.Adam(model.parameters(), lr=0.001)

# DataLoader
train_ds = TensorDataset(torch.tensor(X_seq_tr, dtype=torch.float32), torch.tensor(y_seq_tr, dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

print("Training PyTorch CNN-LSTM model...")
model.train()
for epoch in range(10):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        preds = model(batch_x)
        
        # Custom weighted loss to balance
        loss = -torch.mean(weight_tensor * batch_y * torch.log(preds + 1e-7) + (1 - batch_y) * torch.log(1 - preds + 1e-7))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch_y)
    print(f"Epoch {epoch+1}/10, Loss: {epoch_loss / len(X_seq_tr):.4f}")

model.eval()
with torch.no_grad():
    test_x = torch.tensor(X_seq_ts, dtype=torch.float32).to(device)
    test_preds = model(test_x).cpu().numpy()
    y_pred_pytorch = (test_preds > 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_seq_ts, y_pred_pytorch).ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
tss = sensitivity + specificity - 1
print(f"\nCNN-LSTM Metrics on Test set:\nTSS: {tss:.4f}, Sensitivity: {sensitivity:.4f}, Specificity: {specificity:.4f}")

### Step 9: Save Models for Local Telemetry Dashboard
We serialize the trained models and standard scalers. Download these files and place them in your local project workspace `models/` directory.

In [ ]:
os.makedirs('models', exist_ok=True)
xgb_model.save_model('models/xgboost_forecaster.json')
joblib.dump(scaler, 'models/xgboost_forecaster_scaler.joblib')
torch.save(model.state_dict(), 'models/cnn_lstm_model.pt')
print("Successfully saved models! Go ahead and download: ")
print("1. models/xgboost_forecaster.json")
print("2. models/xgboost_forecaster_scaler.joblib")
print("3. models/cnn_lstm_model.pt")